In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Load the dataset
df = pd.read_csv('latin_america_gdp_growth.csv')
# Display the first few rows (as detailed in your proposal's Table 1)
display(df.head())
# Print data information to check for missing values (like YR1960)
print(df.info())

## Data Preprocessing and Exploratory Data Analysis. (EDA)

### 1. Reshaping and Handling Missing Data

We will transpose the dataset so that the rows represent the chronological timeline. Then, we will drop the year 1960 since it has no values for each country.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# 2. Drop the 1960 column completely
if 'YR1960' in df.columns:
    df = df.drop(columns=['YR1960'])
    print("Successfully dropped the YR1960 column.")
# 3. Reshape: Set 'ISO' as index, drop 'Country' name, and transpose
ts_data = df.drop(columns=['Country']).set_index('economy').T
# 4. Clean the index: Convert 'YR1961' onwards to a proper DateTime index
ts_data.index = ts_data.index.str.replace('YR', '').astype(int)
ts_data.index = pd.to_datetime(ts_data.index, format='%Y')
# Verify we have a clean dataset with no missing values
ts_data_clean = ts_data.interpolate(method='time').bfill()
print(f"Remaining Missing Values: {ts_data_clean.isna().sum().sum()}")
print(f"New Time Series Start Year: {ts_data_clean.index.min().year}")
display(ts_data_clean.head())

### 2. Exploratory Data Analysis (EDA)

Let's visualize the economic trajectories of a few major economies. This will help us visually spot periods of high volatility (like the 1980s debt crisis or the 2020 pandemic).

In [ ]:
# Set a consistent plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(14, 6))
# Plotting Brazil, Mexico, and Argentina as examples
plt.plot(ts_data_clean.index, ts_data_clean['BRA'], label='Brazil (BRA)', linewidth=2)
plt.plot(ts_data_clean.index, ts_data_clean['MEX'], label='Mexico (MEX)', linewidth=2)
plt.plot(ts_data_clean.index, ts_data_clean['ARG'], label='Argentina (ARG)', linewidth=2, alpha=0.7)
plt.title('Annual GDP Growth Trajectories (1961 - 2024)', fontsize=14)
plt.ylabel('GDP Growth (Annual %)', fontsize=12)
plt.xlabel('Year', fontsize=12)
plt.axhline(0, color='black', linestyle='--', linewidth=1) # Zero-growth baseline
plt.legend()
plt.tight_layout()
plt.show()

### 3. Stationarity Testing (ADF Test)

ARIMA models require the time series to be stationary (constant mean and variance over time). We will loop through all 20 countries and run the Augmented Dickey-Fuller (ADF) test.

- If the p-value is <0.05, the series is stationary (d=0).
- If the p-value is >0.05, the series has a unit root and requires differencing (d=1).

In [ ]:
def check_stationarity(timeseries, country_code):
    """Runs ADF test and returns integration order needed."""
    result = adfuller(timeseries)
    p_value = result[1]

    if p_value > 0.05:
        print(f"[{country_code}] p-value: {p_value:.4f} -> Non-Stationary (Requires differencing: d=1)")
        return 1
    else:
        print(f"[{country_code}] p-value: {p_value:.4f} -> Stationary (d=0)")
        return 0
# Dictionary to store the required differencing order for the ARIMA phase
integration_orders = {}
print("--- Augmented Dickey-Fuller Test Results ---")
for country in ts_data_clean.columns:
    integration_orders[country] = check_stationarity(ts_data_clean[country], country)

### 4. ACF and PACF Analysis

To figure out the Autoregressive (p) and Moving Average (q) terms for our baseline ARIMA models, we look at the ACF and PACF plots. This block automatically applies differencing if the ADF test in the previous step determined it was necessary.

In [ ]:
# Pick a specific country to analyze (you can change this to 'BRA', 'MEX', etc.)
target_country = 'HND'
series = ts_data_clean[target_country]
# Apply differencing if the ADF test deemed it necessary
if integration_orders[target_country] > 0:
    series = series.diff().dropna()
    print(f"Applied 1st order differencing to {target_country} before plotting.")
else:
    print(f"{target_country} is stationary. No differencing applied.")
# Plot ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(series, ax=axes[0], lags=20, title=f'Autocorrelation (ACF) - {target_country}')
plot_pacf(series, ax=axes[1], lags=20, title=f'Partial Autocorrelation (PACF) - {target_country}', method='ywm')
plt.tight_layout()
plt.show()

## Statistical Modelling (ARIMA)

We will use the pmdarima library to run an auto_arima search. This will automatically test different combinations of p and q and select the best model for each country based on the lowest Akaike Information Criterion (AIC).

In [ ]:
import pmdarima as pm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
# 1. Define Train/Test Horizons
test_size = 5 # Years 2020 to 2024
train_data = ts_data_clean.iloc[:-test_size]
test_data = ts_data_clean.iloc[-test_size:]
# Dictionaries to store results and predictions
arima_results_cv = {}
arima_forecasts_cv = pd.DataFrame(index=test_data.index)
print("--- Fitting Auto-ARIMA with Rolling Origin Validation ---")
# 2. Loop through all 20 countries
for country in ts_data_clean.columns:
    print(f"Running Rolling Origin CV for {country}...")

    # Initial fit on the training set (1961 - 2019)
    model = pm.auto_arima(train_data[country],
                          start_p=0, start_q=0,
                          max_p=4, max_q=4,
                          d=None,
                          seasonal=False,
                          trace=False,
                          error_action='ignore',
                          suppress_warnings=True,
                          stepwise=True)

    rolling_forecasts = []

    # 3. Walk-Forward (Rolling Origin) Loop for the 5 test years
    for i in range(len(test_data)):
        # Predict exactly 1 step ahead (FIX: Using np.asarray to handle both Series and Arrays safely)
        prediction = model.predict(n_periods=1)
        pred_value = np.asarray(prediction)[0]
        rolling_forecasts.append(pred_value)

        # Get the actual observation that just "occurred"
        actual_obs = test_data[country].iloc[i]

        # Update the model with the new observation before predicting the next year
        model.update(actual_obs)

    arima_forecasts_cv[country] = rolling_forecasts

    # 4. Calculate Evaluation Metrics
    rmse = np.sqrt(mean_squared_error(test_data[country], rolling_forecasts))
    mae = mean_absolute_error(test_data[country], rolling_forecasts)

    # Store results
    arima_results_cv[country] = {
        'ARIMA Order (p,d,q)': model.order,
        'RMSE': rmse,
        'MAE': mae
    }
# 5. Display the final metrics sorted by RMSE
baseline_metrics_cv_df = pd.DataFrame(arima_results_cv).T
print("\n--- Rolling Origin CV Evaluation Metrics ---")
display(baseline_metrics_cv_df.sort_values(by='RMSE'))

In [ ]:
# Select a few countries to visualize the rolling forecast vs actuals
countries_to_plot = ['ARG', 'BRA', 'MEX']
plt.figure(figsize=(15, 6))
for country in countries_to_plot:
    # Plot Actuals
    plt.plot(ts_data_clean.index[-15:], ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o')

    # Plot Rolling Origin Forecasts
    plt.plot(test_data.index, arima_forecasts_cv[country],
             label=f'Rolling Forecast {country}', linestyle='--', marker='X', markersize=8)
plt.title('ARIMA Rolling Origin Forecasts vs Actual GDP Growth (2010 - 2024)', fontsize=14)
plt.axvline(x=2019.5, color='black', linestyle=':', label='Start of Rolling Origin (Train/Test Split)')
plt.ylabel('GDP Growth (%)')
plt.xlabel('Year')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Machine Learning Modelling

### 1. XGBoost

Instead of fitting 20 isolated local models, we are going to build a Global Supervised Model using an ensemble tree algorithm (we'll use XGBoost).

To do this, we need to completely restructure our dataset:
- **Long Format (Panel Data):** We will stack all countries into a single dataset.
- **Feature Engineering:** Machine learning models don't naturally "understand" time. We have to create explicit lag features (e.g., GDP of t−1, t−2, t−3) so the model can look at the recent past to predict the future.
- **Global Learning:** By pooling the data, the model learns regional interactions. For instance, it can learn how a shock in larger economies might correlate with trends across the rest of the continent.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
# 1. Prepare Panel Data (Convert to Long Format)
long_df = ts_data_clean.reset_index().rename(columns={'index': 'Year'})
# FIX 1: Extract just the integer year from the DateTime objects
long_df['Year'] = long_df['Year'].dt.year
long_df = long_df.melt(id_vars=['Year'], var_name='Country', value_name='GDP_Growth')
# Sort to ensure chronological order per country
long_df = long_df.sort_values(['Country', 'Year']).reset_index(drop=True)
# 2. Feature Engineering: Create Lag Features
num_lags = 3 # We will use the past 3 years to predict the current year
for i in range(1, num_lags + 1):
    long_df[f'Lag_{i}'] = long_df.groupby('Country')['GDP_Growth'].shift(i)
# Drop the first 3 years per country since they now have NaN lags
panel_df = long_df.dropna().reset_index(drop=True)
# 3. Rolling Origin CV Setup
test_years = [2020, 2021, 2022, 2023, 2024]
xgb_forecasts_cv = pd.DataFrame(index=test_years, columns=ts_data_clean.columns)
print("--- Training Global XGBoost Model with Rolling Origin CV ---")
for year in test_years:
    print(f"Training up to {year-1}, predicting {year}...")

    # Train set: All data strictly BEFORE the current test year
    train_df = panel_df[panel_df['Year'] < year]
    # Test set: Exactly the current test year
    test_df = panel_df[panel_df['Year'] == year]

    # Extract Features (X) and Target (y)
    X_train = pd.get_dummies(train_df.drop(columns=['Year', 'GDP_Growth']), columns=['Country'])
    y_train = train_df['GDP_Growth']

    X_test = pd.get_dummies(test_df.drop(columns=['Year', 'GDP_Growth']), columns=['Country'])

    # Align columns to ensure the test set has exactly the same features as the train set
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    # Initialize and Train the Model
    xgb_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)

    # Predict for all 20 countries simultaneously
    preds = xgb_model.predict(X_test)

    # Store predictions dynamically
    for i, country in enumerate(test_df['Country'].values):
        xgb_forecasts_cv.at[year, country] = preds[i]
# 4. Calculate Final Evaluation Metrics
xgb_results = {}
# FIX 2: Convert the integer test years back into DateTime objects to query the actuals correctly
test_dates = pd.to_datetime([str(y) for y in test_years])
for country in ts_data_clean.columns:
    actuals = ts_data_clean.loc[test_dates, country]
    preds = xgb_forecasts_cv[country].astype(float)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)

    xgb_results[country] = {'RMSE': rmse, 'MAE': mae}
# Display metrics alongside ARIMA for comparison
xgb_metrics_df = pd.DataFrame(xgb_results).T
print("\n--- XGBoost Rolling CV Evaluation Metrics ---")
display(xgb_metrics_df.sort_values(by='RMSE'))

In [ ]:
countries_to_plot = ['ARG', 'BRA', 'MEX']
plt.figure(figsize=(15, 6))
for country in countries_to_plot:
    # Actual Data (Extract integer years for the x-axis)
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2)

    # ARIMA Forecast (from Phase 2)
    plt.plot(test_years, arima_forecasts_cv[country],
             label=f'ARIMA {country}', linestyle='--', marker='X', alpha=0.6)

    # XGBoost Forecast
    plt.plot(test_years, xgb_forecasts_cv[country],
             label=f'XGBoost {country}', linestyle='-.', marker='s', linewidth=2)
plt.title('ARIMA vs Global XGBoost Forecasts (2010 - 2024)', fontsize=14)
plt.axvline(x=2019.5, color='black', linestyle=':', label='Start of Rolling Origin')
plt.ylabel('GDP Growth (%)')
plt.xlabel('Year')
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 2. Random Forest

While XGBoost builds trees sequentially to correct previous errors (which can sometimes overfit noisy macro data), Random Forest builds independent trees and averages them, which makes it exceptionally robust against variance.

Since we already set up the panel data pipeline, we can swap the algorithm seamlessly while maintaining the exact same Rolling Origin Cross-Validation framework to ensure the RMSE scores are perfectly comparable.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
# 1. Prepare Panel Data (We'll recreate it just to ensure the environment is fresh)
long_df = ts_data_clean.reset_index().rename(columns={'index': 'Year'})
long_df['Year'] = long_df['Year'].dt.year
long_df = long_df.melt(id_vars=['Year'], var_name='Country', value_name='GDP_Growth')
long_df = long_df.sort_values(['Country', 'Year']).reset_index(drop=True)
# Feature Engineering: 3-year lags
num_lags = 3
for i in range(1, num_lags + 1):
    long_df[f'Lag_{i}'] = long_df.groupby('Country')['GDP_Growth'].shift(i)
panel_df = long_df.dropna().reset_index(drop=True)
# 2. Rolling Origin CV Setup
test_years = [2020, 2021, 2022, 2023, 2024]
rf_forecasts_cv = pd.DataFrame(index=test_years, columns=ts_data_clean.columns)
print("--- Training Global Random Forest Model with Rolling Origin CV ---")
for year in test_years:
    print(f"Training up to {year-1}, predicting {year}...")

    train_df = panel_df[panel_df['Year'] < year]
    test_df = panel_df[panel_df['Year'] == year]

    X_train = pd.get_dummies(train_df.drop(columns=['Year', 'GDP_Growth']), columns=['Country'])
    y_train = train_df['GDP_Growth']

    X_test = pd.get_dummies(test_df.drop(columns=['Year', 'GDP_Growth']), columns=['Country'])
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    # Initialize and Train Random Forest
    # max_depth=5 helps prevent individual trees from memorizing the noise in the panel data
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    preds = rf_model.predict(X_test)

    for i, country in enumerate(test_df['Country'].values):
        rf_forecasts_cv.at[year, country] = preds[i]
# 3. Calculate Final Evaluation Metrics
rf_results = {}
test_dates = pd.to_datetime([str(y) for y in test_years])
for country in ts_data_clean.columns:
    actuals = ts_data_clean.loc[test_dates, country]
    preds = rf_forecasts_cv[country].astype(float)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)

    rf_results[country] = {'RMSE': rmse, 'MAE': mae}
# Display metrics
rf_metrics_df = pd.DataFrame(rf_results).T
print("\n--- Random Forest Rolling CV Evaluation Metrics ---")
display(rf_metrics_df.sort_values(by='RMSE'))

In [ ]:
countries_to_plot = ['ARG', 'BRA', 'MEX']
plt.figure(figsize=(15, 6))
for country in countries_to_plot:
    # Actual Data
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2, color='black')

    # XGBoost Forecast (From previous step)
    plt.plot(test_years, xgb_forecasts_cv[country],
             label=f'XGBoost {country}', linestyle='--', marker='s', alpha=0.7)

    # Random Forest Forecast
    plt.plot(test_years, rf_forecasts_cv[country],
             label=f'Random Forest {country}', linestyle='-.', marker='^', linewidth=2)
plt.title('Machine Learning Forecasts: Random Forest vs XGBoost (2010 - 2024)', fontsize=14)
plt.axvline(x=2019.5, color='gray', linestyle=':', label='Start of Rolling Origin')
plt.ylabel('GDP Growth (%)')
plt.xlabel('Year')
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Functional Time Series (Scalar-on-Function Regression)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pywt # The PyWavelets library
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
# 1. Define the functional window size
window_size = 5 # Use the trajectory of the past 5 years
# Setup Rolling Origin CV
test_years = [2020, 2021, 2022, 2023, 2024]
fts_forecasts_cv = pd.DataFrame(index=test_years, columns=ts_data_clean.columns)
# Choose the Wavelet Family
wavelet_family = 'haar'
print(f"--- Training Functional Time Series Model (Wavelet: {wavelet_family}) ---")
for year in test_years:
    print(f"Training Wavelet FTS up to {year-1}, predicting {year}...")

    train_df = ts_data_clean.loc[:str(year-1)]

    X_train_list = []
    y_train_list = []

    # 2. Extract rolling functional windows for the training set
    for country in ts_data_clean.columns:
        series = train_df[country].values
        for i in range(len(series) - window_size):
            window = series[i : i + window_size]

            # 3. Apply Discrete Wavelet Transform (DWT)
            # This decomposes the 5-year curve into Trend (cA) and Shock (cD) coefficients
            coeffs = pywt.wavedec(window, wavelet_family, level=1)

            # Flatten the coefficients into a single feature array
            wavelet_features = np.concatenate(coeffs)

            X_train_list.append(wavelet_features)
            y_train_list.append(series[i + window_size])

    X_train_arr = np.array(X_train_list)
    y_train_arr = np.array(y_train_list)

    # 4. Train the Scalar-on-Function Regression Model
    fts_model = LinearRegression()
    fts_model.fit(X_train_arr, y_train_arr)

    # 5. Predict the Test Year
    for country in ts_data_clean.columns:
        # Get exactly the 'window_size' years leading up to the test year
        window = train_df[country].iloc[-window_size:].values

        # Transform the test window into wavelet space
        coeffs = pywt.wavedec(window, wavelet_family, level=1)
        wavelet_features = np.concatenate(coeffs).reshape(1, -1)

        # Generate the prediction
        pred = fts_model.predict(wavelet_features)[0]
        fts_forecasts_cv.at[year, country] = pred
# 6. Calculate Final Evaluation Metrics
fts_results = {}
test_dates = pd.to_datetime([str(y) for y in test_years])
for country in ts_data_clean.columns:
    actuals = ts_data_clean.loc[test_dates, country]
    preds = fts_forecasts_cv[country].astype(float)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)

    fts_results[country] = {'RMSE': rmse, 'MAE': mae}
# Display metrics
fts_metrics_df = pd.DataFrame(fts_results).T
print("\n--- Functional (Wavelet) Time Series Evaluation Metrics ---")
display(fts_metrics_df.sort_values(by='RMSE'))

In [ ]:
countries_to_plot = ['ARG']
plt.figure(figsize=(16, 7))
for country in countries_to_plot:
    # Actual Data
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2, color='black')

    # ARIMA (Statistical)
    plt.plot(test_years, arima_forecasts_cv[country],
             label=f'ARIMA {country}', linestyle=':', marker='X', alpha=0.6)

    # Random Forest (Machine Learning)
    plt.plot(test_years, rf_forecasts_cv[country],
             label=f'RF {country}', linestyle='--', marker='^', alpha=0.8)

    # FTS (Functional Data Analysis)
    plt.plot(test_years, fts_forecasts_cv[country],
             label=f'FTS {country}', linestyle='-', marker='s', linewidth=2)
plt.title('Methodological Comparison: Statistical vs ML vs Functional Forecasts (2010 - 2024)', fontsize=15)
plt.axvline(x=2019.5, color='gray', linestyle='-.', label='Start of Rolling CV (Crisis Horizon)')
plt.ylabel('GDP Growth (%)', fontsize=12)
plt.xlabel('Year', fontsize=12)
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
countries_to_plot = ['BRA']
plt.figure(figsize=(16, 7))
for country in countries_to_plot:
    # Actual Data
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2, color='black')

    # ARIMA (Statistical)
    plt.plot(test_years, arima_forecasts_cv[country],
             label=f'ARIMA {country}', linestyle=':', marker='X', alpha=0.6)

    # Random Forest (Machine Learning)
    plt.plot(test_years, rf_forecasts_cv[country],
             label=f'RF {country}', linestyle='--', marker='^', alpha=0.8)

    # FTS (Functional Data Analysis)
    plt.plot(test_years, fts_forecasts_cv[country],
             label=f'FTS {country}', linestyle='-', marker='s', linewidth=2)
plt.title('Methodological Comparison: Statistical vs ML vs Functional Forecasts (2010 - 2024)', fontsize=15)
plt.axvline(x=2019.5, color='gray', linestyle='-.', label='Start of Rolling CV (Crisis Horizon)')
plt.ylabel('GDP Growth (%)', fontsize=12)
plt.xlabel('Year', fontsize=12)
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
countries_to_plot = ['MEX']
plt.figure(figsize=(16, 7))
for country in countries_to_plot:
    # Actual Data
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2, color='black')

    # ARIMA (Statistical)
    plt.plot(test_years, arima_forecasts_cv[country],
             label=f'ARIMA {country}', linestyle=':', marker='X', alpha=0.6)

    # Random Forest (Machine Learning)
    plt.plot(test_years, rf_forecasts_cv[country],
             label=f'RF {country}', linestyle='--', marker='^', alpha=0.8)

    # FTS (Functional Data Analysis)
    plt.plot(test_years, fts_forecasts_cv[country],
             label=f'FTS {country}', linestyle='-', marker='s', linewidth=2)
plt.title('Methodological Comparison: Statistical vs ML vs Functional Forecasts (2010 - 2024)', fontsize=15)
plt.axvline(x=2019.5, color='gray', linestyle='-.', label='Start of Rolling CV (Crisis Horizon)')
plt.ylabel('GDP Growth (%)', fontsize=12)
plt.xlabel('Year', fontsize=12)
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
countries_to_plot = ['ECU']
plt.figure(figsize=(16, 7))
for country in countries_to_plot:
    # Actual Data
    actual_years = ts_data_clean.index[-15:].year
    plt.plot(actual_years, ts_data_clean[country].iloc[-15:],
             label=f'Actual {country}', marker='o', linewidth=2, color='black')

    # ARIMA (Statistical)
    plt.plot(test_years, arima_forecasts_cv[country],
             label=f'ARIMA {country}', linestyle=':', marker='X', alpha=0.6)

    # Random Forest (Machine Learning)
    plt.plot(test_years, rf_forecasts_cv[country],
             label=f'RF {country}', linestyle='--', marker='^', alpha=0.8)

    # FTS (Functional Data Analysis)
    plt.plot(test_years, fts_forecasts_cv[country],
             label=f'FTS {country}', linestyle='-', marker='s', linewidth=2)
plt.title('Methodological Comparison: Statistical vs ML vs Functional Forecasts (2010 - 2024)', fontsize=15)
plt.axvline(x=2019.5, color='gray', linestyle='-.', label='Start of Rolling CV (Crisis Horizon)')
plt.ylabel('GDP Growth (%)', fontsize=12)
plt.xlabel('Year', fontsize=12)
# Clean up legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Diebold-Mariano Test Validation

How to read the results:
- **P-Value:** If this is <0.05, you can reject the null hypothesis. It means the difference in performance is statistically significant.
- **DM Statistic:** If it is a positive number, it means the errors of Model 1 were larger than Model 2, so Model 2 wins. If it is a negative number, it means Model 1 wins.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")
def diebold_mariano_test(actual, pred1, pred2, h=1):
    """
    Computes the Diebold-Mariano test statistic.
    Null Hypothesis: Two forecasts have the same accuracy.
    """
    actual = np.asarray(actual)
    pred1 = np.asarray(pred1)
    pred2 = np.asarray(pred2)

    # Calculate forecast errors
    e1 = actual - pred1
    e2 = actual - pred2

    # Calculate loss differential (using Squared Error loss)
    d = (e1 ** 2) - (e2 ** 2)
    d_bar = np.mean(d)

    N = len(d)

    # Calculate autocovariance of the loss differential
    gamma = np.zeros(h)
    for i in range(h):
        gamma[i] = np.sum((d[i:] - d_bar) * (d[:N-i] - d_bar)) / N

    # Variance of d
    var_d = gamma[0] + 2 * np.sum(gamma[1:])

    if var_d == 0:
        return 0, 1.0 # Variance is 0, cannot reject null

    # Calculate DM Statistic and p-value
    dm_stat = d_bar / np.sqrt(var_d / N)
    p_value = 2 * (1 - norm.cdf(abs(dm_stat)))

    return dm_stat, p_value
print("--- Running Diebold-Mariano Tests ---")
# We will collect the global errors (concatenating all 20 countries)
# to see which model performed statistically best overall for Latin America.
global_actuals = []
global_arima = []
global_ml = [] # Change to xgb_forecasts_cv if you preferred XGBoost!
global_fts = []
test_dates = pd.to_datetime([str(y) for y in test_years])
for country in ts_data_clean.columns:
    global_actuals.extend(ts_data_clean.loc[test_dates, country].values)
    global_arima.extend(arima_forecasts_cv[country].values)

    # Note: Using rf_forecasts_cv here. If you used XGBoost, change this to xgb_forecasts_cv
    global_ml.extend(rf_forecasts_cv[country].values)
    global_fts.extend(fts_forecasts_cv[country].values)
# 1. ARIMA vs Machine Learning
dm_stat_1, p_val_1 = diebold_mariano_test(global_actuals, global_arima, global_ml)
# 2. ARIMA vs Functional Time Series
dm_stat_2, p_val_2 = diebold_mariano_test(global_actuals, global_arima, global_fts)
# 3. Machine Learning vs Functional Time Series
dm_stat_3, p_val_3 = diebold_mariano_test(global_actuals, global_ml, global_fts)
# Compile the final report
dm_results = pd.DataFrame({
    'Comparison': [
        'ARIMA vs Machine Learning',
        'ARIMA vs Wavelet FTS',
        'Machine Learning vs Wavelet FTS'
    ],
    'DM Statistic': [dm_stat_1, dm_stat_2, dm_stat_3],
    'P-Value': [p_val_1, p_val_2, p_val_3]
})
def interpret_dm(row):
    model_1, model_2 = row['Comparison'].split(' vs ')
    if row['P-Value'] > 0.05:
        return "No statistical difference in accuracy"
    elif row['DM Statistic'] > 0:
        return f"{model_2} is significantly better"
    else:
        return f"{model_1} is significantly better"
dm_results['Conclusion'] = dm_results.apply(interpret_dm, axis=1)
display(dm_results)